# 02 - Exploratory Data Analysis (EDA)
**Enhancing E-Commerce Sales Forecasting Using Google Trends**

---

## Objectives
1. Understand sales patterns across categories
2. Analyze Google Trends seasonality
3. Examine correlation between trends and sales
4. Identify festive-season effects

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Load merged dataset
df = pd.read_csv('../data/processed/merged_sales_trends.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f'Dataset shape: {df.shape}')
print(f'Categories: {df["Category"].unique()}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
df.head()

## 2.1 Sales Distribution by Category

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, cat in enumerate(df['Category'].unique()):
    cat_data = df[df['Category'] == cat]
    axes[i].hist(cat_data['Units_Sold'], bins=25, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{cat}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Units Sold / Week')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(cat_data['Units_Sold'].mean(), color='red', linestyle='--', label=f'Mean: {cat_data["Units_Sold"].mean():.0f}')
    axes[i].legend()

plt.suptitle('Distribution of Weekly Sales by Category', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.2 Sales Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

for cat in df['Category'].unique():
    cat_data = df[df['Category'] == cat].sort_values('Date')
    ax.plot(cat_data['Date'], cat_data['Units_Sold'], label=cat, linewidth=1.5)

# Mark festive seasons
for year in [2022, 2023, 2024]:
    ax.axvspan(pd.Timestamp(f'{year}-10-01'), pd.Timestamp(f'{year}-12-31'), 
               alpha=0.1, color='orange', label='Festive Season' if year == 2022 else '')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Units Sold', fontsize=12)
ax.set_title('Weekly Sales Over Time (All Categories)', fontsize=16, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2.3 Google Trends Time Series

In [ ]:
trend_keywords = ['earbuds india', 'wireless earphones', 'bluetooth earbuds', 
                   'best earbuds under 2000', 'noise earbuds']

# Use one category to avoid duplicated trend rows
cat0 = df['Category'].unique()[0]
trend_data = df[df['Category'] == cat0].sort_values('Date')

fig, ax = plt.subplots(figsize=(16, 6))
for kw in trend_keywords:
    ax.plot(trend_data['Date'], trend_data[kw], label=kw, linewidth=1.5)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Search Interest (0-100)', fontsize=12)
ax.set_title('Google Trends: Earbuds Keywords in India', fontsize=16, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2.4 Correlation: Sales vs Google Trends

In [ ]:
# Correlation matrix for key features
corr_cols = ['Units_Sold', 'Avg_Price'] + trend_keywords
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Matrix: Sales vs Google Trends', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation of each keyword with Units_Sold:')
for kw in trend_keywords:
    r = df['Units_Sold'].corr(df[kw])
    print(f'  {kw:35s} r = {r: .3f}')

## 2.5 Lag Correlation Analysis

**Key question:** Does search interest *lead* sales? If so, by how many weeks?

In [ ]:
lag_cols = [f'earbuds india_lag{i}' for i in range(1, 5)]
lag_corrs = {}

for col in ['earbuds india'] + lag_cols:
    lag_corrs[col] = df['Units_Sold'].corr(df[col])

lag_df = pd.DataFrame(list(lag_corrs.items()), columns=['Feature', 'Correlation'])

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(lag_df['Feature'], lag_df['Correlation'], color=['tab:blue'] + ['tab:orange']*4, edgecolor='black')
ax.set_ylabel('Correlation with Units_Sold')
ax.set_title('Lag Correlation: "earbuds india" Search vs Sales', fontsize=14, fontweight='bold')
ax.set_xticklabels(lag_df['Feature'], rotation=30, ha='right')

for bar, val in zip(bars, lag_df['Correlation']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', 
            ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

best_lag = lag_df.loc[lag_df['Correlation'].abs().idxmax()]
print(f'\nBest predictor: {best_lag["Feature"]} (r = {best_lag["Correlation"]:.3f})')

## 2.6 Festive Season vs. Non-Festive Analysis

In [ ]:
festive = df.groupby('Is_Festive_Season')['Units_Sold'].agg(['mean', 'std', 'count']).round(1)
festive.index = ['Non-Festive (Jan-Sep)', 'Festive (Oct-Dec)']
print('Sales by Season:')
print(festive)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by season
df['Season'] = df['Is_Festive_Season'].map({0: 'Non-Festive', 1: 'Festive (Oct-Dec)'})
sns.boxplot(data=df, x='Season', y='Units_Sold', ax=axes[0], palette='Set2')
axes[0].set_title('Sales by Season', fontsize=14, fontweight='bold')

# Monthly box plot
sns.boxplot(data=df, x='Month', y='Units_Sold', ax=axes[1], palette='viridis')
axes[1].set_title('Sales by Month', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 2.7 Summary Statistics

In [ ]:
summary = df.groupby('Category').agg(
    Weeks=('Units_Sold', 'count'),
    Avg_Sales=('Units_Sold', 'mean'),
    Std_Sales=('Units_Sold', 'std'),
    Min_Sales=('Units_Sold', 'min'),
    Max_Sales=('Units_Sold', 'max'),
    Avg_Price=('Avg_Price', 'mean'),
).round(1)

print('Category Summary:')
summary

---

## Key Findings

1. **Seasonality**: Clear spike in Oct-Dec (Indian festive season)
2. **Google Trends correlation**: Search interest correlates with sales
3. **Lag effect**: Lagged search data can predict future sales
4. **Category variation**: Different categories have different sales volumes

**Next:** Open `03_feature_engineering.ipynb` for feature preparation

---